In [17]:
import os
from pathlib import Path
import glob
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # required for .rio methods
import rasterio
import geopandas as gpd
from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------- USER PATHS (raw strings) ----------------------
ROOT_IN  = r"C:\Drought\ESA Soil Moisture"
SHP_DIR  = r"C:\Drought\India Shapefile\indiashapefile"
OUT_DIR  = r"C:\Drought\Regridding and data clipping\SM"
OUT_FILE = "SM_monthly_India_0p05deg_2000-2023.nc"
# ---------------------------------------------------------------------

# Grid / CRS
CRS_EPSG = 4326
TARGET_RES_DEG = 0.05   # 0.05°
FORCE_BOUNDS = None     # e.g., (68.0, 6.0, 98.0, 38.0) for strict bbox

# Chunking / compression
CHUNKS = {"time": 240, "lat": 800, "lon": 800}
COMPRESSION = dict(zlib=True, complevel=4)

# ---------------------- Helpers ----------------------
def find_shp(shp_dir: str) -> Path:
    cand = list(Path(shp_dir).glob("*.shp"))
    if not cand:
        raise FileNotFoundError(f"No .shp file found in: {shp_dir}")
    return cand[0]

def _detect_spatial_dims(dims):
    """Return (lon_dim, lat_dim) from a dims Mapping, being generous about names."""
    lon_dim = None
    lat_dim = None
    for d in dims:
        dl = d.lower()
        if lon_dim is None and ("lon" in dl or dl in ("x", "longitude", "nlon")):
            lon_dim = d
        if lat_dim is None and ("lat" in dl or dl in ("y", "latitude", "nlat")):
            lat_dim = d
    return lon_dim, lat_dim

def promote_coord(ds: xr.Dataset, name: str) -> xr.Dataset:
    """If a variable exists but isn't a coordinate, promote it to a coordinate."""
    if name in ds.variables and name not in ds.coords:
        ds = ds.set_coords(name)
    return ds

def normalize_coords_and_dims(ds: xr.Dataset) -> xr.Dataset:
    """
    Make sure longitude/latitude are:
      - dimension names literally 'lon' and 'lat' (rename dims if needed)
      - present as coordinates (promote if only variables)
      - ascending order
    """
    # 1) First, if coords are named 'longitude'/'latitude', rename to lon/lat.
    rename_vars = {}
    if "longitude" in ds.variables: rename_vars["longitude"] = "lon"
    if "latitude"  in ds.variables: rename_vars["latitude"]  = "lat"
    if rename_vars:
        ds = ds.rename(rename_vars)

    # 2) Detect current dim names for lon/lat and rename dims to 'lon'/'lat'
    lon_dim, lat_dim = _detect_spatial_dims(ds.dims)
    rename_dims = {}
    if lon_dim and lon_dim != "lon": rename_dims[lon_dim] = "lon"
    if lat_dim and lat_dim != "lat": rename_dims[lat_dim] = "lat"
    if rename_dims:
        ds = ds.rename(rename_dims)

    # 3) Ensure lon/lat are coordinates (not just variables)
    if "lon" in ds.dims:
        ds = promote_coord(ds, "lon")
    if "lat" in ds.dims:
        ds = promote_coord(ds, "lat")

    if "lon" in ds.dims and "lon" not in ds.coords and "lon" in ds.variables:
        ds = ds.assign_coords(lon=ds["lon"])
    if "lat" in ds.dims and "lat" not in ds.coords and "lat" in ds.variables:
        ds = ds.assign_coords(lat=ds["lat"])

    # 4) Sort ascending
    if "lat" in ds.coords and ds.coords["lat"].values[0] > ds.coords["lat"].values[-1]:
        ds = ds.sortby("lat")
    if "lon" in ds.coords and ds.coords["lon"].values[0] > ds.coords["lon"].values[-1]:
        ds = ds.sortby("lon")

    return ds

def ensure_spatial(obj):
    """
    Prepare a DataArray/Dataset for rioxarray:
      - ensure dims are literally lon/lat (rename if needed)
      - ensure lon/lat coords exist and are ascending
      - write EPSG:4326 CRS
      - set spatial dims: x=lon, y=lat
    """
    if isinstance(obj, xr.Dataset):
        ds = normalize_coords_and_dims(obj)
        ds = ds.rio.write_crs(CRS_EPSG, inplace=False)
        ds = ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
        return ds
    else:
        # DataArray → Dataset → fix → back to DataArray
        da = obj
        ds = da.to_dataset(name="_tmp_da_")
        ds = ensure_spatial(ds)
        return ds["_tmp_da_"]

def pick_sm_var(ds: xr.Dataset) -> str:
    for v in ["sm", "soil_moisture", "ssm", "sm_surface"]:
        if v in ds.data_vars:
            return v
    if len(ds.data_vars) == 1:
        return list(ds.data_vars)[0]
    raise KeyError(f"Could not find soil moisture var in: {list(ds.data_vars)}")

def month_end(resampled):
    t = pd.to_datetime(resampled["time"].values)
    return resampled.assign_coords(time=(t + pd.offsets.MonthEnd(0)).values)

def bounds_from_shape(gdf: gpd.GeoDataFrame, pad_deg=0.1):
    minx, miny, maxx, maxy = gdf.to_crs(epsg=CRS_EPSG).total_bounds
    return (minx - pad_deg, miny - pad_deg, maxx + pad_deg, maxy + pad_deg)

def build_template(bounds, res_deg=TARGET_RES_DEG) -> xr.Dataset:
    minx, miny, maxx, maxy = bounds
    lons = np.arange(minx, maxx + 1e-12, res_deg)
    lats = np.arange(miny, maxy + 1e-12, res_deg)
    ds = xr.Dataset(coords={"lon": ("lon", lons), "lat": ("lat", lats)})
    return ensure_spatial(ds)

def clip_to_shape(da_or_ds, gdf: gpd.GeoDataFrame):
    da_or_ds = ensure_spatial(da_or_ds)
    gdf = gdf.to_crs(epsg=CRS_EPSG)
    return da_or_ds.rio.clip(gdf.geometry, gdf.crs, drop=True, invert=False)

def regrid_to_template(da_or_ds, template: xr.Dataset):
    da_or_ds = ensure_spatial(da_or_ds)
    template  = ensure_spatial(template)
    # rioxarray needs geotransform on the template; add a dummy var if none present
    if len(template.data_vars) == 0:
        template = template.assign(
            _dummy=(("lat","lon"), np.zeros((template.sizes["lat"], template.sizes["lon"])))
        )
        template = ensure_spatial(template)
    out = da_or_ds.rio.reproject_match(template, resampling=rasterio.enums.Resampling.bilinear)
    if "_dummy" in out:
        out = out.drop_vars("_dummy")
    return out

# ---------------------- Core processing ----------------------
def process_year(year: int, india_gdf: gpd.GeoDataFrame, template: xr.Dataset) -> xr.Dataset | None:
    year_dir = Path(ROOT_IN) / str(year)
    files = sorted(glob.glob(str(year_dir / "ESACCI-SOILMOISTURE-L3S-SSMV-COMBINED-*.nc")))
    if not files:
        print(f"[{year}] No files found. Skipping.")
        return None

    # Auto-detect engine; robust to netcdf4/h5netcdf/scipy
    ds = xr.open_mfdataset(files, combine="by_coords", parallel=False)
    ds = ensure_spatial(ds)  # <-- normalizes dims+coords and sets spatial dims

    var = pick_sm_var(ds)   # usually "sm"
    da  = ds[var]

    # Quick bbox subset using template bounds (faster) then precise clip
    minx, miny, maxx, maxy = template.rio.bounds()
    da = da.sel(lon=slice(minx, maxx), lat=slice(miny, maxy))

    # Daily → monthly mean (month-end timestamps)
    da_m = da.resample(time="MS").mean(keep_attrs=True)
    da_m = month_end(da_m)
    da_m = ensure_spatial(da_m)

    # Clip to India
    da_clip = clip_to_shape(da_m, india_gdf)

    # Regrid to 0.05°
    da_rg = regrid_to_template(da_clip, template)

    # Attributes
    da_rg.name = "sm"
    da_rg.attrs.update({
        "long_name": "Surface soil moisture",
        "units": da.attrs.get("units", "m3 m-3"),
        "source": "ESA CCI Soil Moisture L3S SSM COMBINED",
        "processing_lineage": "daily→monthly mean; clipped to India; regridded to 0.05° (bilinear); timestamps at month-end",
    })

    out = da_rg.to_dataset(name="sm")
    out = ensure_spatial(out)
    # CF niceties
    out["lat"].attrs.update({"standard_name": "latitude", "units": "degrees_north"})
    out["lon"].attrs.update({"standard_name": "longitude", "units": "degrees_east"})
    out["time"].attrs.update({"standard_name": "time"})
    return out

def main():
    # India shape & template grid
    shp_path = find_shp(SHP_DIR)
    india_gdf = gpd.read_file(shp_path)

    bounds = FORCE_BOUNDS if FORCE_BOUNDS is not None else bounds_from_shape(india_gdf, pad_deg=0.1)
    template = build_template(bounds, res_deg=TARGET_RES_DEG)

    # Years present
    year_dirs = sorted([p for p in Path(ROOT_IN).iterdir() if p.is_dir() and p.name.isdigit()])
    years = [int(p.name) for p in year_dirs if 2000 <= int(p.name) <= 2023]
    if not years:
        raise RuntimeError(f"No year folders (2000–2023) found in {ROOT_IN}")

    # Process
    yearly = []
    for y in tqdm(years, desc="Processing years"):
        try:
            dsy = process_year(y, india_gdf, template)
            if dsy is not None:
                yearly.append(dsy)
        except Exception as e:
            print(f"[{y}] ERROR: {e}")

    if not yearly:
        raise RuntimeError("No yearly datasets were produced.")

    # Concatenate on time
    ds_all = xr.concat(yearly, dim="time").sortby("time")

    # Optional chunking
    try:
        ds_all = ds_all.chunk(CHUNKS)
    except Exception:
        pass

    # Encoding & global attrs
    encoding = {v: {**COMPRESSION} for v in ds_all.data_vars}
    ds_all.attrs.update({
        "title": "ESA CCI Soil Moisture – Monthly mean, India, 0.05°",
        "summary": "Daily L3S SSM COMBINED aggregated to monthly mean, clipped to India, regridded to 0.05° regular lat/lon.",
        "Conventions": "CF-1.8",
        "crs": f"EPSG:{CRS_EPSG}",
        "history": "Created by sm_monthly_india_0p05_all_years.py",
        "spatial_resolution": "0.05 degree",
        "temporal_aggregation": "monthly mean (month-end timestamps)",
    })

    os.makedirs(OUT_DIR, exist_ok=True)
    out_path = str(Path(OUT_DIR) / OUT_FILE)

    # Save with backend fallback
    try:
        ds_all.to_netcdf(out_path, format="NETCDF4", engine="netcdf4", encoding=encoding)
    except Exception:
        try:
            ds_all.to_netcdf(out_path, engine="h5netcdf", encoding=encoding)
        except Exception:
            ds_all.to_netcdf(out_path, engine="scipy", encoding=encoding)

    print(f"\nSaved single file → {out_path}")

if __name__ == "__main__":
    main()


Processing years: 100%|██████████| 23/23 [09:52<00:00, 25.77s/it]



Saved single file → C:\Drought\Regridding and data clipping\SM\SM_monthly_India_0p05deg_2000-2023.nc


In [35]:
import os
from pathlib import Path
import glob
import warnings

import numpy as np
import xarray as xr
import rioxarray  # enables .rio.* methods
import rasterio
import geopandas as gpd

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------- USER PATHS ----------------------
IN_DIR   = r"C:\Drought\GLDAS"
SHP_DIR  = r"C:\Drought\India Shapefile\indiashapefile"
OUT_DIR  = r"C:\Drought\Regridding and data clipping\SM"
OUT_FILE = "GLDAS_NOAH_monthly_India_0p05deg_2000-2023.nc"
# -------------------------------------------------------

CRS_EPSG = 4326
TARGET_RES_DEG = 0.05
FORCE_BOUNDS = None   # e.g., (68.0, 6.0, 98.0, 38.0) to force exact bbox

# Keep default key variables; set KEEP_ALL_VARS=True to keep everything
KEEP_ALL_VARS = False
VARS_TO_KEEP = [
    "Evap_tavg",
    "SoilMoi0_10cm_inst",
    "SoilMoi10_40cm_inst",
    "SoilMoi40_100cm_inst",
    "SoilMoi100_200cm_inst",
]

# IO tuning
CHUNKS = {"time": 300, "lat": 800, "lon": 800}
COMPRESSION = dict(zlib=True, complevel=4)

# ---------------------- Helpers ----------------------
def find_shp(shp_dir: str) -> Path:
    cand = list(Path(shp_dir).glob("*.shp"))
    if not cand:
        raise FileNotFoundError(f"No .shp file found in: {shp_dir}")
    return cand[0]

def _detect_spatial_dims(dims):
    lon_dim = None
    lat_dim = None
    for d in dims:
        dl = d.lower()
        if lon_dim is None and ("lon" in dl or dl in ("x", "longitude", "nlon")):
            lon_dim = d
        if lat_dim is None and ("lat" in dl or dl in ("y", "latitude", "nlat")):
            lat_dim = d
    return lon_dim, lat_dim

def promote_coord(ds: xr.Dataset, name: str) -> xr.Dataset:
    if name in ds.variables and name not in ds.coords:
        ds = ds.set_coords(name)
    return ds

def normalize_coords_and_dims(ds: xr.Dataset) -> xr.Dataset:
    # 1) rename variables named latitude/longitude to lat/lon
    rename_vars = {}
    if "longitude" in ds.variables: rename_vars["longitude"] = "lon"
    if "latitude"  in ds.variables: rename_vars["latitude"]  = "lat"
    if rename_vars:
        ds = ds.rename(rename_vars)

    # 2) rename dims to lon/lat if they have other names
    lon_dim, lat_dim = _detect_spatial_dims(ds.dims)
    rename_dims = {}
    if lon_dim and lon_dim != "lon": rename_dims[lon_dim] = "lon"
    if lat_dim and lat_dim != "lat": rename_dims[lat_dim] = "lat"
    if rename_dims:
        ds = ds.rename(rename_dims)

    # 3) ensure lon/lat are coordinates
    if "lon" in ds.dims:
        ds = promote_coord(ds, "lon")
    if "lat" in ds.dims:
        ds = promote_coord(ds, "lat")
    if "lon" in ds.dims and "lon" not in ds.coords and "lon" in ds.variables:
        ds = ds.assign_coords(lon=ds["lon"])
    if "lat" in ds.dims and "lat" not in ds.coords and "lat" in ds.variables:
        ds = ds.assign_coords(lat=ds["lat"])

    # 4) sort ascending for clean slices
    if "lat" in ds.coords and ds.coords["lat"].values[0] > ds.coords["lat"].values[-1]:
        ds = ds.sortby("lat")
    if "lon" in ds.coords and ds.coords["lon"].values[0] > ds.coords["lon"].values[-1]:
        ds = ds.sortby("lon")
    return ds

def ensure_spatial(obj):
    """
    Prepare DataArray/Dataset for rioxarray:
    - dims are lon/lat; coords exist and ascending
    - write EPSG:4326 CRS
    - set spatial dims x=lon, y=lat
    - preserve variable name for DataArray
    """
    if isinstance(obj, xr.Dataset):
        ds = normalize_coords_and_dims(obj)
        ds = ds.rio.write_crs(CRS_EPSG, inplace=False)
        ds = ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
        return ds
    else:
        da = obj
        orig_name = da.name if da.name is not None else "var"
        ds = da.to_dataset(name=orig_name)
        ds = ensure_spatial(ds)
        return ds[orig_name]

def bounds_from_shape(gdf: gpd.GeoDataFrame, pad_deg=0.1):
    minx, miny, maxx, maxy = gdf.to_crs(epsg=CRS_EPSG).total_bounds
    return (minx - pad_deg, miny - pad_deg, maxx + pad_deg, maxy + pad_deg)

def build_template(bounds, res_deg=TARGET_RES_DEG) -> xr.Dataset:
    minx, miny, maxx, maxy = bounds
    lons = np.arange(minx, maxx + 1e-12, res_deg)
    lats = np.arange(miny, maxy + 1e-12, res_deg)
    ds = xr.Dataset(coords={"lon": ("lon", lons), "lat": ("lat", lats)})
    return ensure_spatial(ds)

def clip_to_shape(da_or_ds, gdf: gpd.GeoDataFrame):
    da_or_ds = ensure_spatial(da_or_ds)
    gdf = gdf.to_crs(epsg=CRS_EPSG)
    return da_or_ds.rio.clip(gdf.geometry, gdf.crs, drop=True, invert=False)

def regrid_to_template(da_or_ds, template: xr.Dataset):
    da_or_ds = ensure_spatial(da_or_ds)
    template  = ensure_spatial(template)
    # rioxarray needs geotransform on the template; add a dummy var if none present
    if len(template.data_vars) == 0:
        template = template.assign(
            _dummy=(("lat","lon"), np.zeros((template.sizes["lat"], template.sizes["lon"])))
        )
        template = ensure_spatial(template)
    out = da_or_ds.rio.reproject_match(template, resampling=rasterio.enums.Resampling.bilinear)
    if "_dummy" in out:
        out = out.drop_vars("_dummy")
    return out

# ---------------------- Pipeline ----------------------
def main():
    # India shape & template grid
    shp_path = find_shp(SHP_DIR)
    india_gdf = gpd.read_file(shp_path)

    bounds = FORCE_BOUNDS if FORCE_BOUNDS is not None else bounds_from_shape(india_gdf, pad_deg=0.1)
    template = build_template(bounds, res_deg=TARGET_RES_DEG)

    # Discover monthly GLDAS files
    files = sorted(glob.glob(str(Path(IN_DIR) / "GLDAS_NOAH025_M.A*.nc4.SUB.nc4")))
    if not files:
        raise RuntimeError(f"No GLDAS files found in {IN_DIR}")

    # Open all months into one dataset
    ds = xr.open_mfdataset(files, combine="by_coords", parallel=False)
    ds = ensure_spatial(ds)

    # Keep selected vars (or all)
    if not KEEP_ALL_VARS:
        keep = [v for v in VARS_TO_KEEP if v in ds.data_vars]
        if not keep:
            raise RuntimeError(f"None of the expected variables found. Available: {list(ds.data_vars)}")
        ds = ds[keep]

    # Quick bbox subset (speeds up clip)
    minx, miny, maxx, maxy = template.rio.bounds()
    ds = ds.sel(lon=slice(minx, maxx), lat=slice(miny, maxy))

    # Clip & regrid each variable
    out_vars = []
    for v in ds.data_vars:
        da = ds[v]
        da = clip_to_shape(da, india_gdf)
        da = regrid_to_template(da, template)
        da = da.rename(v)  # ensure stable, unique name
        da.attrs.update({
            "processing_lineage": "monthly; clipped to India; regridded to 0.05° (bilinear)",
            "source": "GLDAS Noah 0.25° (monthly)",
        })
        out_vars.append(da)

    # Merge variables; override minor coord attr diffs if any
    out = xr.merge([d.to_dataset(name=d.name) for d in out_vars], compat="override")
    out = ensure_spatial(out)

    # Chunking & compression
    try:
        out = out.chunk(CHUNKS)
    except Exception:
        pass
    encoding = {v: {**COMPRESSION} for v in out.data_vars}

    # Global metadata
    out.attrs.update({
        "title": "GLDAS Noah monthly – India 0.05°",
        "summary": "GLDAS Noah 0.25° monthly regridded to 0.05°, clipped to India.",
        "Conventions": "CF-1.8",
        "crs": f"EPSG:{CRS_EPSG}",
        "history": "Created by gldas_monthly_india_0p05_all_years.py",
        "spatial_resolution": "0.05 degree",
        "temporal_aggregation": "monthly",
    })

    # Save (robust backend fallback)
    os.makedirs(OUT_DIR, exist_ok=True)
    out_path = str(Path(OUT_DIR) / OUT_FILE)
    try:
        out.to_netcdf(out_path, format="NETCDF4", engine="netcdf4", encoding=encoding)
    except Exception:
        try:
            out.to_netcdf(out_path, engine="h5netcdf", encoding=encoding)
        except Exception:
            out.to_netcdf(out_path, engine="scipy", encoding=encoding)

    print(f"\nSaved → {out_path}")

if __name__ == "__main__":
    main()



Saved → C:\Drought\Regridding and data clipping\SM\GLDAS_NOAH_monthly_India_0p05deg_2000-2023.nc
